In [ ]:
# 1. Genesisのインストール
!pip install -q genesis-world

import genesis as gs
import numpy as np

gs.init(backend=gs.cuda)

scene = gs.Scene(show_viewer=False)

# Franka Emika Panda
plane = scene.add_entity(gs.morphs.Plane())
franka = scene.add_entity(
    gs.morphs.MJCF(file='xml/franka_emika_panda/panda.xml'),
)

# カメラ
cam = scene.add_camera(
    res=(640, 480),
    pos=(2.5, 1.5, 1.5),
    lookat=(0.0, 0.0, 0.5),
    fov=40,
)

scene.build()

cam.start_recording()

for i in range(120):
    # 簡単なサイン波で関節の角度を指定して動かすテスト
    qpos = np.zeros(franka.n_qs)
    qpos[1] = 0.5 * np.sin(i * 0.05)
    qpos[3] = -1.5 + 0.5 * np.cos(i * 0.05)
    franka.set_qpos(qpos)

    scene.step()
    cam.render()

# 保存
cam.stop_recording(save_to_filename='test_robot.mp4', fps=30)
print("動画の保存が完了しました！")

In [ ]:
import inspect
print(inspect.signature(cam.stop_recording))


In [ ]:
import genesis as gs
import numpy as np
import imageio
from IPython.display import Video

try:
    gs.init(backend=gs.cuda)
except Exception:
    print("Genesisはすでに初期化済みなのでそのまま進みます！")

scene = gs.Scene(show_viewer=False)

# Franka Emika Panda
plane = scene.add_entity(gs.morphs.Plane())
franka = scene.add_entity(
    gs.morphs.MJCF(file='xml/franka_emika_panda/panda.xml'),
)

# カメラ
cam = scene.add_camera(
    res=(640, 480),
    pos=(2.5, 1.5, 1.5),
    lookat=(0.0, 0.0, 0.5),
    fov=40,
)

scene.build()

frames = []

for i in range(120):
    qpos = np.zeros(franka.n_qs)
    qpos[1] = 0.5 * np.sin(i * 0.05)
    qpos[3] = -1.5 + 0.5 * np.cos(i * 0.05)
    franka.set_qpos(qpos)

    scene.step()

    render_out = cam.render()
    rgb = render_out[0] if isinstance(render_out, tuple) else render_out
    frames.append(rgb)

# 保存
imageio.mimsave(
    'test_robot_mac.mp4',
    frames,
    fps=30,
    codec='libx264',
    pixelformat='yuv420p'
)
print("Mac対応の動画（test_robot_mac.mp4）を保存しました！")

Video('test_robot_mac.mp4', embed=True, width=600)

In [ ]:
# 1. 必要なライブラリのインストール
!pip install -q transformers torch pillow opencv-python

import genesis as gs
import numpy as np
import imageio
import cv2
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
from IPython.display import Video

# 2. 視覚言語AIモデル（CLIP）の読み込み
print("AIモデル（CLIP）")
model_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(model_id)
clip_processor = CLIPProcessor.from_pretrained(model_id)

# 3. Genesisの初期化
try:
    gs.init(backend=gs.cuda)
except Exception:
    pass

scene = gs.Scene(show_viewer=False)

# 床とロボットアームを配置
plane = scene.add_entity(gs.morphs.Plane())
franka = scene.add_entity(
    gs.morphs.MJCF(file='xml/franka_emika_panda/panda.xml'),
)

# 色をすべて統一し、形を変える
common_surface = gs.surfaces.Default(color=(0.95, 0.55, 0.15, 1.0))

# ① 球体（Sphere）: 右側
obj_sphere = scene.add_entity(
    gs.morphs.Sphere(radius=0.055, pos=(0.45, -0.25, 0.055), fixed=True),
    surface=common_surface
)

# ② 立方体（Box）: 中央
obj_box = scene.add_entity(
    gs.morphs.Box(size=(0.09, 0.09, 0.09), pos=(0.50, 0.00, 0.045), fixed=True),
    surface=common_surface
)

# ③ 円柱（Cylinder）: 左側
obj_cylinder = scene.add_entity(
    gs.morphs.Cylinder(radius=0.045, height=0.14, pos=(0.45, 0.25, 0.07), fixed=True),
    surface=common_surface
)

# カメラ配置
cam = scene.add_camera(
    res=(640, 480),
    pos=(1.45, 0.0, 0.85),
    lookat=(0.35, 0.0, 0.15),
    fov=45,
)

scene.build()

# 4. AIによる形状推論関数（3つの領域を切り出してテキスト指示と照合）
def ai_identify_shape(rgb_img, target_prompt):
    h, w, _ = rgb_img.shape
    # カメラ画像内の3つのオブジェクト領域を切り出し
    crops = [
        Image.fromarray(rgb_img[int(h*0.48):int(h*0.85), int(w*0.55):int(w*0.85)]), # 球体エリア
        Image.fromarray(rgb_img[int(h*0.48):int(h*0.85), int(w*0.40):int(w*0.60)]), # 立方体エリア
        Image.fromarray(rgb_img[int(h*0.42):int(h*0.85), int(w*0.15):int(w*0.45)]), # 円柱エリア
    ]
    # 形状の候補テキストと比較して推論
    candidate_texts = [
        target_prompt,
        "empty flat grey floor background"
    ]
    inputs = clip_processor(text=candidate_texts, images=crops, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = clip_model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)[:, 0].numpy()
    best_idx = int(np.argmax(probs))
    confidence = float(probs[best_idx])
    return best_idx, confidence

# 5. タスクのシナリオ定義（形状テキストで4つのミッションを実行＝70秒）
missions = [
    {"prompt": "a round sphere ball",      "name": "SPHERE (Ball)",     "yaw": -0.48},
    {"prompt": "a square cube block",      "name": "CUBE (Box)",        "yaw":  0.00},
    {"prompt": "a tall cylinder pillar",   "name": "CYLINDER (Pillar)", "yaw":  0.48},
    {"prompt": "a round sphere ball (2nd)","name": "SPHERE (Ball)",     "yaw": -0.48},
]

# 初期姿勢（ホームポジション）
home_qpos = np.array([0.0, -0.4, 0.0, -1.8, 0.0, 1.5, 0.7, 0.04, 0.04])
franka.set_qpos(home_qpos)
scene.step()

frames = []
print("形状認識AIシミュレーションと録画を開始します（約2分かかります）...")

for m_i, m in enumerate(missions):
    # カメラ画像を取得してAI（CLIP）で形状を推論
    render_out = cam.render()
    rgb_now = render_out[0] if isinstance(render_out, tuple) else render_out
    pred_idx, conf = ai_identify_shape(rgb_now, m["prompt"])

    # ターゲットへ向かう関節角度
    reach_qpos = np.array([m["yaw"], 0.55, 0.0, -1.35, 0.0, 1.9, 0.7, 0.0, 0.0])

    # 525フレーム × 4ミッション = 2100フレーム（30fpsで70秒）
    for step in range(525):
        if step < 90:
            qpos = home_qpos
            status = "Phase 1: AI (CLIP) Analyzing 3D Geometry..."
        elif step < 240:
            t = (step - 90) / 150.0
            smooth_t = 0.5 - 0.5 * np.cos(np.pi * t)
            qpos = (1 - smooth_t) * home_qpos + smooth_t * reach_qpos
            status = f"Phase 2: Approaching Shape [{m['name']}]"
        elif step < 375:
            qpos = reach_qpos.copy()
            qpos[6] += 0.4 * np.sin((step - 240) * 0.15)
            status = f"Phase 3: Inspecting Shape [{m['name']}]"
        else:
            t = (step - 375) / 150.0
            smooth_t = 0.5 - 0.5 * np.cos(np.pi * t)
            qpos = (1 - smooth_t) * reach_qpos + smooth_t * home_qpos
            status = "Phase 4: Returning to Home Position"

        franka.set_qpos(qpos)
        scene.step()

        render_out = cam.render()
        rgb = render_out[0] if isinstance(render_out, tuple) else render_out

        # 画面にAIの形状推論ステータスと倍速表記を描画
        img_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        cv2.rectangle(img_bgr, (10, 10), (630, 115), (20, 20, 20), -1)
        cv2.putText(img_bgr, f"Mission {m_i+1}/4 | Shape Prompt: '{m['prompt']}'", (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(img_bgr, f"CLIP Shape Recognition: [{m['name']}] (Score: {conf:.2f})", (20, 65),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.50, (100, 255, 100), 1, cv2.LINE_AA)
        cv2.putText(img_bgr, status, (20, 95),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.53, (100, 220, 255), 1, cv2.LINE_AA)
        cv2.putText(img_bgr, "2x Speed", (530, 460),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2, cv2.LINE_AA)

        frames.append(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))

# 6. MP4動画
output_filename = 'final_shape_robot.mp4'
imageio.mimsave(
    output_filename,
    frames,
    fps=30,
    codec='libx264',
    pixelformat='yuv420p'
)
print(f"形状認識版の完成動画（{output_filename}・長さ {len(frames)/30:.1f}秒）")

# 再生確認
Video(output_filename, embed=True, width=600)